In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
DATA_PATH = next(
    (root / "data/raw/creditcard.csv" for root in (Path.cwd(), *Path.cwd().parents)
     if (root / "data/raw/creditcard.csv").exists()),
    None,
)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find data/raw/creditcard.csv in the current directory or its parents."
    )

df = pd.read_csv(DATA_PATH)
df = df.sort_values("Time").reset_index(drop=True)

train_end = int(len(df) * 0.60)
validation_end = int(len(df) * 0.80)

train_df = df.iloc[:train_end].copy()
validation_df = df.iloc[train_end:validation_end].copy()
test_df = df.iloc[validation_end:].copy()

FEATURES = [column for column in df.columns if column != "Class"]

X_train = train_df[FEATURES]
y_train = train_df["Class"]

X_validation = validation_df[FEATURES]
y_validation = validation_df["Class"]

X_test = test_df[FEATURES]
y_test = test_df["Class"]

print(f"Training fraud cases: {y_train.sum():,}")
print(f"Validation fraud cases: {y_validation.sum():,}")
print(f"Test fraud cases: {y_test.sum():,}")

In [ ]:
scale_features = ["Time", "Amount"]
pca_features = [f"V{i}" for i in range(1, 29)]

preprocessor = ColumnTransformer(
    transformers=[
        ("scaled", RobustScaler(), scale_features),
        ("pca", "passthrough", pca_features),
    ]
)

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=2000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

logistic_model.fit(X_train, y_train)

logistic_scores = logistic_model.predict_proba(
    X_validation
)[:, 1]

In [ ]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()
class_ratio = negative_count / positive_count

print(f"Legitimate training cases: {negative_count:,}")
print(f"Fraud training cases: {positive_count:,}")
print(f"Negative-to-positive ratio: {class_ratio:.2f}")

In [ ]:
xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    n_estimators=400,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=class_ratio,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

xgb_model.fit(X_train, y_train)

xgb_scores = xgb_model.predict_proba(
    X_validation
)[:, 1]

In [ ]:
xgb_unweighted = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    n_estimators=400,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

xgb_unweighted.fit(X_train, y_train)

xgb_unweighted_scores = xgb_unweighted.predict_proba(
    X_validation
)[:, 1]

In [ ]:
isolation_preprocessor = ColumnTransformer(
    transformers=[
        ("scaled", RobustScaler(), ["Time", "Amount"]),
        ("pca", "passthrough", pca_features),
    ]
)

X_train_transformed = isolation_preprocessor.fit_transform(X_train)
X_validation_transformed = isolation_preprocessor.transform(X_validation)

normal_train = X_train_transformed[y_train.to_numpy() == 0]

isolation_model = IsolationForest(
    n_estimators=300,
    max_samples="auto",
    contamination="auto",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

isolation_model.fit(normal_train)

In [ ]:
isolation_scores = -isolation_model.score_samples(
    X_validation_transformed
)

In [ ]:
def evaluate_scores(y_true, scores, model_name):
    return {
        "model": model_name,
        "average_precision": average_precision_score(y_true, scores),
        "roc_auc": roc_auc_score(y_true, scores),
    }


results = pd.DataFrame(
    [
        evaluate_scores(
            y_validation,
            logistic_scores,
            "Class-weighted logistic regression",
        ),
        evaluate_scores(
            y_validation,
            xgb_unweighted_scores,
            "Unweighted XGBoost",
        ),
        evaluate_scores(
            y_validation,
            xgb_scores,
            "Class-weighted XGBoost",
        ),
        evaluate_scores(
            y_validation,
            isolation_scores,
            "Isolation Forest",
        ),
    ]
).sort_values("average_precision", ascending=False)

results

In [ ]:
def precision_recall_at_k(y_true, scores, k):
    y_array = np.asarray(y_true)
    scores_array = np.asarray(scores)

    ranked_indices = np.argsort(scores_array)[::-1]
    top_indices = ranked_indices[:k]

    fraud_detected = y_array[top_indices].sum()
    total_fraud = y_array.sum()

    return {
        "k": k,
        "fraud_detected": int(fraud_detected),
        "precision_at_k": fraud_detected / k,
        "recall_at_k": fraud_detected / total_fraud,
    }